[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/cv-internship-in-a-book/blob/main/notebooks/week11_fairness_audit_STARTER.ipynb)


# Week 11 -- Fairness Audit — Who Does the Model Fail? (STARTER)
### The Computer Vision Internship | MediVision AI Lagos

**This week:** fairness audit — sex gap on chest X-ray, magnification gap on BreakHis, full fairness brief

**Dataset:** Multi-domain (fairness audit)

**STARTER notebook** -- your working environment. Complete each exercise before moving on.


In [ ]:
# -- Colab/Local Setup -- run this first in Colab, skip locally -------------
import os

if not os.path.exists('requirements.txt'):
    # Clone the full course repository
    !git clone https://github.com/internshipinabook/cv-internship-in-a-book.git cv-book
    %cd cv-book
    # Install all required packages (suppress verbose output with -q)
    !pip install -r requirements.txt -q

# Create working directories
os.makedirs('outputs',  exist_ok=True)   # charts, reports, predictions
os.makedirs('models',   exist_ok=True)   # trained model checkpoints
os.makedirs('datasets', exist_ok=True)   # raw downloaded data
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds --------------------------------------------------
# Setting seeds ensures every random operation produces the same result
# on every run. Essential for: comparing runs, debugging, sharing results.

import random    # Python built-in random
import numpy as np   # NumPy random (data loading, sklearn)
import torch     # PyTorch random (neural networks, dropout)

SEED = 42        # 42 is conventional -- any integer works

random.seed(SEED)           # fix Python random() calls
np.random.seed(SEED)        # fix np.random.* calls
torch.manual_seed(SEED)     # fix PyTorch CPU operations

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)  # fix GPU operations too

# Prefer deterministic cuDNN algorithms (may slow training ~5%)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Seeds fixed: {SEED} | Device: {device}')


## Learning Objectives

By the end of this week, you will be able to:

- Conduct a sex-disaggregated performance audit on the chest X-ray model
- Measure the magnification fairness gap on the BreakHis model
- Identify the contributing site with the largest performance gap in the TCGA model
- Apply threshold adjustment as a mitigation strategy and measure its before/after effect
- Write a complete NLP fairness brief suitable for a hospital ethics board



---

## MONDAY -- "Sex-Disaggregated Audit — Chest X-Ray"


The NIH Chest X-Ray14 metadata includes Patient_Gender. Compute AUC separately for male and female patients for all 14 conditions. Report: aggregate AUC (what the original paper reported), male AUC, female AUC, and the sex gap for each condition. Which conditions show the largest gap? Is the gap systematic or condition-specific?


### Exercise 11.1 -- Sex-disaggregated AUC table

For all 14 chest X-ray conditions: compute male AUC, female AUC, and the sex gap. Sort by gap size. Which condition has the largest gap? Translate into a clinical sentence.


In [ ]:
# Exercise 11.1: Sex-disaggregated AUC table
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Monday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Monday: Sex-Disaggregated Audit — Chest X-Ray (scaffold) --
import pandas as pd
from sklearn.metrics import roc_auc_score

meta = pd.read_csv("datasets/xray/Data_Entry_2017.csv")

# Merge predictions with patient metadata
results_df = pd.DataFrame({"image_id": test_ids, "y_score": predictions})
results_df = results_df.merge(meta[["Image Index","Patient Gender"]], 
                               left_on="image_id", right_on="Image Index")

for cond_idx, cond_name in enumerate(CONDITIONS):
    for gender in ["M", "F"]:
        mask = results_df["Patient Gender"] == gender
        if mask.sum() > 0 and y_true[mask, cond_idx].sum() > 0:
            auc = roc_auc_score(y_true[mask, cond_idx], results_df.loc[mask, "y_score"])
            print(f"  {cond_name:<25} {gender}: AUC = {auc:.4f}")


### Monday Morning Moment

*Slack — Monday, 9:00am.*

**Ngozi Eze-Okafor:** The Cardiomegaly AUC gap between male and female patients is 0.09. The original NIH paper reported 0.90 aggregate AUC. For female patients it is 0.86.

**Dr. Kwame Asante:** Is cardiomegaly more or less common in female patients?

**Ngozi Eze-Okafor:** More common, actually. Women have higher rates of heart failure with preserved ejection fraction.

**Dr. Kwame Asante:** So the most at-risk group is also the worst-served group.

**Ngozi Eze-Okafor:** Yes. The model underperforms on female patients for the condition they are more likely to have.

**Dr. Kwame Asante:** That is the sentence that goes in the fairness brief. Write it exactly as you said it.




---

## TUESDAY -- "Magnification Fairness — BreakHis"


The magnification gap you measured in Week 7 is a fairness gap. A lab that uses 400× magnification is systematically underserved by a model trained at 40×. The patients whose slides are processed at 400× receive worse-quality predictions — not because they are different patients, but because their lab uses a different protocol.


### Exercise 11.2 -- Fairness metric comparison

Compute three fairness metrics for the sex-disaggregated chest X-ray audit: (a) Demographic parity gap (prediction rate difference), (b) Equal opportunity gap (recall difference), (c) AUC gap. Which metric would an ethics board find most actionable? Why?


In [ ]:
# Exercise 11.2: Fairness metric comparison
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Tuesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Tuesday: Magnification Fairness — BreakHis (scaffold) --
# Reframe the Week 7 magnification gap as a fairness audit
# "Group" = magnification level (proxy for lab protocol)
mag_groups = {"40X": [], "100X": [], "200X": [], "400X": []}
for mag, preds, labels in magnification_results:
    auc = roc_auc_score(labels, preds)
    gap = auc_40x - auc
    print(f"  {mag}: AUC = {auc:.4f} | Gap from 40x: {gap:+.4f}")
    # Clinical translation:
    # "Patients at labs using {mag} magnification receive
    #  predictions that are {gap*100:.1f} percentage points
    #  less accurate than patients at 40x labs."



---

## WEDNESDAY -- "Threshold Mitigation — Adjusting for Group Fairness"


One mitigation strategy: adjust the classification threshold separately for each group to equalise sensitivity (recall). For the sex-disaggregated chest X-ray audit, find the threshold that achieves equal sensitivity for Pneumonia detection in male and female patients. Measure the cost: how does per-group precision change?


### Exercise 11.3 -- Threshold adjustment effect

Apply threshold adjustment to equalise Pneumonia sensitivity across sex. Report: sensitivity before/after for each group, specificity before/after, aggregate AUC before/after. Is the mitigation worth the precision cost?


In [ ]:
# Exercise 11.3: Threshold adjustment effect
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Wednesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Wednesday: Threshold Mitigation — Adjusting for Group Fairness (scaffold) --
from sklearn.metrics import precision_recall_curve

def find_threshold_for_sensitivity(y_true, y_score, target_sensitivity=0.80):
    """Find the decision threshold that achieves the target sensitivity."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    # Find threshold closest to target sensitivity
    idx = np.argmin(np.abs(recall - target_sensitivity))
    return thresholds[idx], recall[idx], precision[idx]

# Find equalised thresholds per sex
for gender in ["M", "F"]:
    mask = results_df["Patient Gender"] == gender
    thresh, sens, prec = find_threshold_for_sensitivity(
        y_true[mask, PNEUMONIA_IDX], 
        y_score[mask], 
        target_sensitivity=0.80)
    print(f"  {gender}: threshold={thresh:.3f}, sensitivity={sens:.3f}, precision={prec:.3f}")



---

## THURSDAY -- "The Fairness Brief for the Ethics Board"


A hospital ethics board is not a data science audience. The fairness brief must answer: who does the model fail for, how much worse, what are the clinical consequences, and what does the team recommend. The methodology is an appendix. The recommendation is the first paragraph.


### Exercise 11.4 -- Residual risk quantification

After threshold adjustment: what is the remaining performance gap for the Cardiomegaly condition? For a hospital with 10,000 female patients per year, how many additional missed Cardiomegaly diagnoses does the residual gap represent?


In [ ]:
# Exercise 11.4: Residual risk quantification
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Thursday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Thursday: The Fairness Brief for the Ethics Board (scaffold) --
# Fairness brief structure (ethics board format):
# Section 1: Summary recommendation (1 paragraph)
#   "We recommend conditional approval with monitoring for..."
# Section 2: Fairness findings (3 bullet points)
#   - Sex gap: Pneumonia detection is X% less sensitive in female patients
#   - Magnification gap: Labs using 400x receive Y% less accurate predictions
#   - Site gap: TCGA site Z shows the largest performance drop
# Section 3: Mitigation applied and its effect
# Section 4: Residual risk after mitigation
# Section 5: Monitoring requirements for deployment
# Methodology appendix



---

## FRIDAY -- "The Friday Build: The Ethics Board Presentation"


Present the fairness brief to Dr. Kwame and Nurse Folake as if they were the ethics board. 15 minutes. Lead with the recommendation. Nurse Folake will ask which patients are most at risk. Dr. Kwame will ask what evidence you have that mitigation worked.


**Friday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Friday: The Friday Build: The Ethics Board Presentation (scaffold) --
# The ethics board presentation:
# Slide 1: Recommendation (30 seconds)
# Slides 2-4: Three fairness findings (3 minutes each)
# Slide 5: Mitigation applied and evidence of effect
# Slide 6: Residual risk
# Slide 7: Monitoring requirements
# Slide 8: Deployment conditions


### Friday Workplace Moment

*MediVision AI — Friday, 2:00pm. Ethics board simulation.*

**Nurse Folake Balogun:** Which patients are most at risk from this model?

**Ngozi Eze-Okafor:** Female patients with Cardiomegaly: the model is 9 AUC points less accurate. Patients at hospitals using 400× magnification for breast cancer slides: 24 AUC points less accurate than 40× labs. Patients at the two smallest TCGA contributing sites: 15 AUC points below the best-performing site.

**Nurse Folake Balogun:** That's three separate groups the model fails.

**Dr. Kwame Asante:** Each requires a different mitigation. Sex gap: threshold adjustment. Magnification gap: per-magnification models or magnification-conditioned training. Site gap: site-specific fine-tuning with local data.

**Nurse Folake Balogun:** Can we deploy with all three mitigations in place?

**Ngozi Eze-Okafor:** We can deploy with threshold adjustment implemented now. The other two require additional data and retraining — minimum 90 days.

**Dr. Kwame Asante:** Conditional approval with a 90-day retraining commitment. Write the deployment recommendation.



## YOUR WEEK 11 CHECKLIST

Check each item before moving to the next week.
If you cannot explain an item without notes, revisit that section.

- [ ] I can state the sex gap for Cardiomegaly detection from memory and give its clinical interpretation.
- [ ] I computed three distinct fairness metrics and can explain which is most appropriate for a clinical ethics board.
- [ ] I applied threshold adjustment and measured its before/after effect on sensitivity and precision.
- [ ] My fairness brief leads with a recommendation — not with methodology.
- [ ] I can quantify the residual risk in terms of missed diagnoses per 10,000 patients.


## Challenge Task

> **Core path:** Implement equalised odds (equal true positive rate AND equal false positive rate across groups). Compare to equal opportunity (equal TPR only). What additional constraint does equalised odds add, and what is its cost in terms of aggregate AUC?

> **Advanced path:** The Fairness Impossibility Theorem (Chouldechova, 2017): it is mathematically impossible to satisfy calibration, false positive rate parity, and false negative rate parity simultaneously when base rates differ. Verify this empirically using your chest X-ray model. Which constraint must be violated given your dataset's sex-specific base rates?


## Common Mistakes

**Reporting only aggregate AUC:** Aggregate AUC of 0.90 hides a 0.09 sex gap on Cardiomegaly. Disaggregated reporting is the minimum for clinical deployment.

**Treating threshold adjustment as a complete mitigation:** Threshold adjustment equalises sensitivity but changes precision. For each group, compute the full confusion matrix before declaring the mitigation effective.

**Framing fairness gaps as model failures rather than data failures:** A sex gap in training data representation produces a sex gap in performance. The model is working correctly on the data it saw. The failure is upstream — in data collection, not in the algorithm.



---

## Get the Full Book

This notebook is part of **The Computer Vision Internship** -- Book 3 of 9, InternshipInABook(tm).

Covers: natural images, medical X-rays, breast & uterine cancer, video, GradCAM/LIME/SHAP/Integrated Gradients, and fairness audits.

Available at [internshipinabook.com](https://internshipinabook.com)
